# NB16 — Insurance Negotiation → Pricing Power (markup over the Medicare floor)

**Place in the five-factor spine (Day 20):** one of the *two* factors that drive the cross-hospital spread (the other is profit margin). Equipment / overhead / labor set the shared floor; **insurance negotiation** moves the ceiling.

**The proxy (Decision 44):** at the procedure level (73721, knee MRI no contrast) we use
`markup = commercial_median / medicare_floor` as the proxy for **pricing power / negotiation leverage**.
This is an *inferred* quantity, NOT a measured profit margin — it carries the "approximation / analysis" label everywhere it appears (Decision 46).

**Structural layer (Decision 45):** each hospital's markup is read against its county funding regime (Parkland / JPS / none) and ownership (public / nonprofit / for-profit). THP-Plano-vs-peers is the natural experiment.

**Dependency note:** the scaffold + synthetic smoke test are DB-independent. **Step R's real markup for Methodist and Parkland rests on the two NB13 medians that are not yet clean** (Methodist MA contamination; Parkland chargemaster basis — Decision 49). Those two rungs stay *gated* until the cleanup lands; MCA / Baylor / THP can populate now.

Mold: Step 0 → 1 → 2 → 3 → R → S → V → Findings → Integrity. Canonical run mode: **Restart & Run All**.


## Step 0 — Source map & constants

- `MEDICARE_FLOOR_73721 = 243.77` (APC 5523, CY2026 OPPS final rule, national unadjusted — the headline benchmark).
- `MEDICARE_FLOOR_73721_DFW = 238.65` (DFW-adjusted, WI ~0.9650 — a *labeled* sensitivity, not the headline).
- Project root / `RAW_DIR` resolver (reuse the NB13 multi-candidate pattern; print the chosen path).
- Confidence flags: `markup_basis = "proxy"`, `floor_confirmed = True`, `medians_publishable = False` (flips True only after Methodist + Parkland fixes).
- Short source map: where the floor came from, where the commercial medians come from (NB13 `per_hospital`).


## Step 1 — Roster: five hospitals × county funding regime × ownership

One row per hospital: short name, CCN (`corroborated`, not yet `confirmed`), county, ownership type, county safety-net regime.

| Hospital | CCN | County | Ownership | Safety net |
|---|---|---|---|---|
| Baylor University Medical Center | 450021 | Dallas | Nonprofit | Parkland |
| Methodist Dallas Medical Center | 450051 | Dallas | Nonprofit | Parkland |
| Parkland Health | 450015 | Dallas | Public | (is the safety net) |
| Texas Health Presbyterian Plano | 450771 | Collin | Nonprofit | **none** |
| Medical City Alliance | 670103 | Tarrant | For-profit | JPS |

Flag anchors; THP (no public hospital in Collin) is the cost-shift test case.


## Step 2 — Markup contract (proxy definition + measured-vs-inferred labels)

Define the metric and its honesty labels in one place:
- `markup_x = commercial_median / MEDICARE_FLOOR_73721`  → pricing-power proxy.
- Spread context: also carry Medicaid / cash rungs if available for the ladder, but **markup is the headline**.
- `markup_basis_confirmed` per hospital: True only where the underlying commercial median is trusted (MCA/Baylor/THP now; Methodist/Parkland after cleanup).
- Loud label: markup is an **approximation of negotiation leverage**, never a profit margin, never per-procedure profitability. NaN sentinel where the median isn't trustworthy — no fabricated markup.


## Step 3 — Synthetic smoke test (DB-independent, watermarked)

Prove the markup math before any real row.
- `SYN_` prefixes on every synthetic hospital; `[SYNTHETIC DATA]` stamped on outputs.
- Plausible commercial medians → markup ladder → spread ratios.
- This cell must run with **no DuckDB files present** (pure in-memory frame).


## Step R — Real wiring: clean `per_hospital` → real markup

Pull the trustworthy commercial medians from NB13 and compute real markup.
- Populate **MCA (906.53), Baylor (1,944.56), THP (1,262.69)** now — clean/confirmed.
- **Gate Methodist and Parkland**: leave markup = NaN with a `pending_cleanup` reason string until Decision 49 fixes land (Methodist MA rows dropped; Parkland basis decided / excluded if %-of-charges).
- No `[SYNTHETIC DATA]` tag once real rows are in; print rows-used and which hospitals are gated.


## Step S — Sensitivity: floor choice & funding-regime grouping

- Recompute markup against the **DFW-adjusted floor** ($238.65) as a labeled sensitivity; report the delta.
- Group markups by **safety-net regime** (Parkland / JPS / none) — does THP (Collin, no public hospital) sit above its nonprofit peers, as the natural experiment predicts?
- Keep this directional; small unbalanced sample (Decision 46) — case study, not inference.


## Step V — Visualization: the markup ladder

- Bar/ladder of markup × per hospital, ordered, with the Medicare floor as the 1.0× baseline.
- Red diagonal `[SYNTHETIC DATA]` watermark whenever the chart is built on synthetic OR partially-gated data.
- **Never post the synthetic or partially-gated chart.** Real chart only after all five rungs are clean.


## Findings — handoff dict

- Emit `NB16_MARKUP = {hospital: {median, markup_x, basis_confirmed, regime, ownership}}`.
- Include a `DO NOT paste into the app / LinkedIn until medians_publishable=True` block.
- One-paragraph plain-language read of the markup ladder + the regime grouping result.


## Integrity — provenance guards

- Block any real markup while `medians_publishable == False` (Methodist/Parkland still gated) from reaching a "publishable" flag.
- Assert synthetic outputs carry `[SYNTHETIC DATA]`; assert no fabricated markup (gated hospitals are NaN, not guessed).
- Print: rows used per hospital, gated count, floor used, `markup_basis="proxy"`. End with **Integrity: OK / BLOCKED**.
